**Machine Learning Lab**

# 7. Regression

**Regression** is a supervised-learning technique used to predict a continuous outcome from one or more predictor variables — for example, predicting a car's fuel efficiency from its weight and horsepower. The most common form is linear regression, valued for its simplicity and interpretability, but there are many other kinds (polynomial, tree-based, etc.), each suited to different data. A regression model's performance is typically judged by metrics like mean squared error (MSE), which measure how far predictions land from the true values.

## 7.1 Importing Modules

In [ ]:
import pandas as pd
import numpy as np

## 7.2 Importing and Preprocessing the Dataset

The [Productivity Prediction of Garment Employees dataset](https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees) will be used for this task. It includes key attributes of the garment manufacturing process and the resulting productivity of employees, collected manually and validated by industry experts.

Attribute information:
```
01  date                  : Date in MM-DD-YYYY
02  day                   : Day of the Week
03  quarter               : A portion of the month. A month was divided into four quarters
04  department            : Associated department with the instance
05  team                  : Associated team number with the instance
06  no_of_workers         : Number of workers in each team
07  no_of_style_change    : Number of changes in the style of a particular product
08  targeted_productivity : Targeted productivity set by the Authority for each team for each day.
09  smv                   : Standard Minute Value, it is the allocated time for a task
10  wip                   : Work in progress. Includes the number of unfinished items for products
11  over_time             : Represents the amount of overtime by each team in minutes
12  incentive             : Represents the amount of financial incentive (in BDT) that enables or motivates a particular course of action.
13  idle_time             : The amount of time when the production was interrupted due to several reasons
14  idle_men              : The number of workers who were idle due to production interruption
15  actual_productivity   : The actual % of productivity that was delivered by the workers. It ranges from 0-1.
```

Run the cell below and upload `garments_worker_productivity.csv` (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select garments_worker_productivity.csv

Load the data and take a look at it.

In [ ]:
# Importing the data
df = pd.read_csv('garments_worker_productivity.csv')
df

Check the unique values in `day`, just to get a feel for the categorical columns.

In [ ]:
df['day'].unique()

Real-world data is often messy in small ways. Checking `df['department'].unique()` reveals `'sweing'`, `'finishing'`, and `'finishing '` — the last one has a trailing space, so pandas (and any encoder downstream) would otherwise treat `'finishing'` and `'finishing '` as two different categories. We strip whitespace from the categorical text columns to fix this before going further.

In [ ]:
# Clean up stray whitespace in text columns (e.g. 'finishing ' vs 'finishing')
for column in ['quarter', 'department', 'day']:
    df[column] = df[column].str.strip()

df['department'].unique()

## 7.3 Train/Test Split

Separate the target we want to predict (`actual_productivity`) from the input features. We select a specific subset of columns as features and drop `date` (a raw date string isn't directly useful to most models without further processing).

In [ ]:
y = df['actual_productivity']
# Feature Selection
X = df[['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']].copy()
X

In [ ]:
from sklearn.model_selection import train_test_split

Hold out 30% of the data for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Check for missing values in the training features before modeling.

In [ ]:
X_train.isnull().sum()

**Task 01**

Use `SimpleImputer` to fill the missing values in the `wip` column with the column mean. Fit-transform on the training data, and only transform (not fit) on the test data — the test set should never influence how the imputer was fitted, or you leak information from test to train.

In [ ]:
# Task 01

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')

X_train['wip'] = imputer.fit_transform(X_train[['wip']])
X_test['wip'] = imputer.transform(X_test[['wip']])

Confirm `wip` no longer has missing values in the training set.

In [ ]:
X_train.isnull().sum()

**Task 02**

Use `LabelEncoder` to encode the categorical columns `quarter`, `department`, and `day` as integers, since scikit-learn models need numeric input.

In [ ]:
# Task 02
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Label Encode only the categorical columns using fit_transform and transform

categorical_columns = ['quarter', 'day', 'department']

for column in categorical_columns:
    X_train[column] = le.fit_transform(X_train[column])
    X_test[column] = le.transform(X_test[column])

Check that the categorical columns are now numeric.

In [ ]:
X_train

**Task 03**

Use `StandardScaler` (or `MinMaxScaler`) to scale the numerical columns so they're on comparable ranges — this matters for models like SVR that are sensitive to feature scale.

In [ ]:
# Task 03
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Normalize only the numerical columns using fit_transform and transform
numerical_columns = ['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

Check the scaled result — the numerical columns should now be roughly centered around 0.

In [ ]:
X_train

## 7.5 Regression

Import four different regression algorithms to compare.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

**Task 04**

Create a dictionary called `regression_models` that maps a descriptive name (string) to each imported regression model instance.

In [ ]:
# Task 04

regression_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regression': DecisionTreeRegressor(random_state=42),
    'Random Forest Regression': RandomForestRegressor(random_state=42),
    'Support Vector Regression': SVR()
}

**Task 05**

Train every model in `regression_models` and measure its mean squared error (MSE) on the test set. Lower MSE means predictions are closer to the true `actual_productivity` values.

In [ ]:
# Task 05

from sklearn.metrics import mean_squared_error
# can be used like this
# mse = mean_squared_error(y_test, y_pred)

for model in regression_models.values():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f'{model}: {mse}')

## 7.6 Practice Problems

1. Besides MSE, compute the R² score (`sklearn.metrics.r2_score`) for each model in `regression_models`. Does the model ranking change compared to using MSE alone?
2. `RandomForestRegressor` exposes a `.feature_importances_` attribute after fitting. Fit it alone, then print the feature importances sorted from most to least important. Which features matter most for predicting productivity?
3. Try `StandardScaler` vs `MinMaxScaler` for the numerical columns in Task 03. Does the choice of scaler noticeably change the MSE for `SVR` (which is scale-sensitive) versus `DecisionTreeRegressor` (which isn't)?
4. The `date` column was dropped entirely in section 7.3. Engineer a new feature from it (e.g. month, or day-of-month) and add it to `X`. Does including it improve any of the models?
5. Re-run Task 01's imputation using `strategy='median'` instead of `strategy='mean'`. Does the final MSE change much? When might median imputation be preferable to mean imputation?